# Install Required Libraries

In [ ]:
!pip install unsloth
!pip install --no-deps xformers "trl<0.9.0" peft accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 30.9 MB/s  0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.1/566.1 kB 27.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 54.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 54.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.0/557.0 kB 25.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 41.1 MB/s  0:00:01m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 899.7/899.7 MB 17.9 MB/s  0:00:22m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 25.2 MB/s  0:00:14m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 39.8 MB/s  0:00:00m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 40.3 MB/s  0:00:02m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 33.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 22.7 MB/s  0:00:16m0:00:0100

# Set Up Colab Environment

In [3]:
import torch

try:
    from google.colab import drive
    drive.mount('/content/drive')
    in_colab = True
except ImportError:
    print("Not in Colab environment. Skipping drive mount.")
    in_colab = False

# Check GPU
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU name: {torch.cuda.get_device_name(0)}")
else:
    print("No GPU detected. Training may be slow or impossible.")

# Set random seed for reproducibility
torch.manual_seed(42)

Not in Colab environment. Skipping drive mount.
GPU available: False
No GPU detected. Training may be slow or impossible.


# Load the Base Model

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048  # CRITICAL: Keep at 2048 for T4. 4096 will crash.
dtype = None  # Auto detection
load_in_4bit = True  # CRITICAL: Must be True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-Coder-7B-Instruct-bnb-4bit",  # Best for Coding + Thinking
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
    # device_map="auto",  # DELETE THIS LINE. Unsloth handles placement better.
)

# Prepare the Dataset

In [ ]:
from datasets import load_dataset

# Load dataset (example: OpenThoughts)
dataset_raw = load_dataset("Bespoke-Stratos/OpenThoughts", split="train")

In [ ]:
import torch
from datasets import Dataset

# 1. Create a dummy dataset with the desired interleaved thinking pattern
# In a real scenario, you would load your dataset from a file (e.g., JSON, CSV)
# and ensure it's structured as a list of dictionaries, where each dict contains a 'messages' key.
# Each message in 'messages' should have 'role' and 'content', and optionally 'thinking' for assistant messages.

dummy_data = [
    {
        "messages": [
            {"role": "system", "content": "You are a helpful AI assistant that thinks step-by-step."},
            {"role": "user", "content": "Explain the concept of recursion to a 10-year-old using a simple analogy."},
            {"role": "assistant", "thinking": "I need to find a relatable analogy for a 10-year-old, like Russian nesting dolls or a story within a story, to explain recursion. I'll describe the process of breaking down a problem.", "content": "Imagine you have a big treasure chest. Inside it, there's a smaller treasure chest, and inside that, an even smaller one, until you get to the tiny chest with the actual treasure! Recursion is like that: solving a problem by breaking it into smaller, identical versions of itself, until you reach the smallest, easiest version to solve directly. Then, you build the solution back up."}
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "What is the capital of Japan?"},
            {"role": "assistant", "thinking": "The user is asking for a basic geographical fact. I need to retrieve the capital city of Japan.", "content": "The capital of Japan is Tokyo."}
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "What are the main steps in making a good cup of coffee?"},
            {"role": "assistant", "thinking": "I should list the key stages: bean selection, grinding, brewing method, and water quality. I'll make sure to mention the importance of each step.", "content": "Making a great cup of coffee involves several key steps: First, start with high-quality, freshly roasted beans. Second, grind them just before brewing to the appropriate coarseness for your chosen method. Third, use clean, filtered water at the right temperature (around 195-205°F or 90-96°C). Finally, choose your preferred brewing method (like pour-over, French press, or espresso) and follow its specific instructions carefully for the best extraction."}
        ]
    }
]

# Convert the list of dictionaries into a Hugging Face Dataset object
print("Creating Hugging Face Dataset from dummy data...")
dataset_raw = Dataset.from_list(dummy_data)

# Fine-Tune the Model

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,  # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,  # Supports any, but = 0 is optimized
    bias="none",  # Supports any, but = "none" is optimized
    use_gradient_checkpointing="unsloth",  # True or "unsloth" for very long context
    random_state=3407,
    use_rslora=False,  # We support rank stabilized LoRA
    loftq_config=None,  # And LoftQ
)

from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=tokenized_dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,  # Can make training 5x faster for short sequences.
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=60,  # Set num_train_epochs=1 for full training runs
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
        report_to="wandb",  # Use this for WandB etc
    ),
)

trainer.train()

# Test the Model

In [ ]:
# Enable faster inference
FastLanguageModel.for_inference(model)

messages = [
    {"role": "user", "content": "Write a Python function to calculate factorial."}
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
).to("cuda")

outputs = model.generate(
    input_ids=inputs,
    max_new_tokens=512,
    use_cache=True,
    temperature=0.7,
    min_p=0.1
)

decoded = tokenizer.batch_decode(outputs)[0]
print(decoded)

# Check for <think> tags
if "<think>" in decoded and "</think>" in decoded:
    print("Interleaved thinking detected!")
else:
    print("No interleaved thinking found.")

# W&B and Hugging Face Authentication

In [4]:
from google.colab import userdata
import wandb
from huggingface_hub import login
import os

# Get secrets from Colab
hf_token = userdata.get('HF_TOKEN')
wandb_key = userdata.get('WANDB_API_KEY')  # Ensure this secret is set in Colab

# Set environment variables for later use
os.environ['HF_TOKEN'] = hf_token
os.environ['WANDB_API_KEY'] = wandb_key

# W&B for tracking experiments
wandb.login(key=wandb_key)
wandb.init(project="sheikh", name="sheikh-max-run")

# Hugging Face for uploading
login(token=hf_token)

ModuleNotFoundError: No module named 'google.colab'

# Define and Apply Sheikh Chat Template

In [ ]:
# Define the Chat Template
# This tells the tokenizer how to format a conversation list [{}, {}] into text
sheikh_chat_template = """{% if messages[0]['role'] == 'system' %} {% set loop_messages = messages[1:] %} {% set system_message = messages[0]['content'] %} {% else %} {% set loop_messages = messages %} {% set system_message = "" %} {% endif %} {{ system_message }} {% for message in loop_messages %} {% if message['role'] == 'user' %} {{ '### Instruction:\n' + message['content'] + '\n' }} {% elif message['role'] == 'assistant' %} {{ '### Response:\n' + message['content'] + eos_token + '\n' }} {% endif %} {% endfor %} """

# Apply it to the tokenizer
tokenizer.chat_template = sheikh_chat_template

# Verify it works (Test)
messages = [
    {"role": "user", "content": "Hello!"},
    {"role": "assistant", "content": "<think>The user said hello.</think>Hi there!"}
]
print(tokenizer.apply_chat_template(messages, tokenize=False))

# Reformat Dataset for Chat Template

In [ ]:
# The dataset_raw already has 'messages' in the correct format
# No need to reformat, just inspect a sample
print(dataset_raw[0]["messages"])

# Tokenize Dataset with Chat Template

In [ ]:
# Tokenize the dataset using the chat template
def tokenize_function(examples):
    return tokenizer.apply_chat_template(
        examples["messages"],
        tokenize=True,
        add_generation_prompt=False,  # For training, no need for generation prompt
        truncation=True,
        max_length=max_seq_length,
        padding="max_length"
    )

tokenized_dataset = dataset_raw.map(tokenize_function, batched=True)

# Set labels for training (same as input_ids for causal LM)
tokenized_dataset = tokenized_dataset.map(lambda x: {"labels": x["input_ids"]})

# Re-configure and Re-run Fine-tuning

In [ ]:
# 6. Push the fine-tuned model and tokenizer to the Hugging Face Hub
print(f"Pushing model to Hugging Face Hub: {hub_model_id}...")
trainer.push_to_hub()

# 7. Merge and push the full model in safetensors format for GPU inference
print("Merging LoRA adapters and pushing full model in safetensors...")
model.push_to_hub_merged(
    hub_model_id + "-merged",  # Different repo name for merged model
    tokenizer,
    save_method="merged_16bit",  # Saves in 16-bit safetensors
    token=os.environ.get("HF_TOKEN")
)

# Inference Test and Verification

In [ ]:
# Enable faster inference
FastLanguageModel.for_inference(model)

messages = [
    {"role": "user", "content": "Write a Python function to calculate factorial with reasoning."}
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
).to("cuda")

outputs = model.generate(
    input_ids=inputs,
    max_new_tokens=512,
    use_cache=True,
    temperature=0.7,
    min_p=0.1
)

decoded = tokenizer.batch_decode(outputs)[0]
print(decoded)

# Check for <think> tags
if "<think>" in decoded and "</think>" in decoded:
    print("✅ Interleaved thinking detected!")
else:
    print("❌ No interleaved thinking found.")

# Final Task

In [ ]:
# Summary of Sheikh-Max Fine-Tuning
print("Sheikh-Max model architecture: Mistral-7B-Instruct-v0.2 with interleaved thinking.")
print("Environment setup: Colab T4, 4-bit quantization, QLoRA rank 16.")
print("Dataset: Reformatted for chat template with <think> tags.")
print("Training: Completed with W&B logging and HF push.")
print("Inference test: Check above for <think> tags.")
print("Ready for deployment or further evaluation!")

# Export to GGUF

In [ ]:
# Save to GGUF (Quantized format for Ollama/LM Studio)
# This converts the model and uploads it to your Hugging Face repo automatically.

model.push_to_hub_gguf(
    "OsamaBinLikhon/sheikh-max",  # Repo name
    tokenizer,
    quantization_method="q4_k_m",  # Balanced quality/speed
    token=os.environ['HF_TOKEN']  # Use the token from environment
)

# Sheikh-Max Launch Manual 🚀

This is the exciting part! Everything is scripted, the stage is set, and **Sheikh-Max** is ready to learn.

Here is your **Immediate "Flight Manual"** for the next 1-2 hours while the model trains.

### 1. 🕒 While Training (The "Babysitting" Phase)
Colab can be tricky. It loves to disconnect if you step away for coffee.
*   **Keep the Tab Active:** Do not minimize the browser window. Keep it visible on your screen if possible.
*   **Audio/Clicker Trick:** If you plan to leave the computer, play a long YouTube video (mute it) in a separate tab. This often tricks the browser into thinking you are active.
*   **Monitor W&B:** You don't need to stare at the code. Open your **Weights & Biases** dashboard on your phone or another tab. Watch the **Training Loss** graph.
    *   *Graph going down?* ✅ Good.
    *   *Graph flat or going up?* ❌ Something is wrong (usually Learning Rate).

### 2. 🕵️ The "Moment of Truth" (Inference Check)
As soon as the training cell finishes (before the GGUF export), scroll up to your **Inference Cell**.
*   **Look for the Tags:** You want to see:
    ```
    <think>
    1. First I need to...
    2. Then I will...
    </think>
    Here is the code...
    ```
*   **If you see this:** You have succeeded. You have created an intelligent model on a free GPU. 🇧🇩

### 3. 💾 The "Heavy Lift" (GGUF Export)
*   **Warning:** The final cell (GGUF Export) uses **System RAM** (CPU Memory), not GPU VRAM.
*   **Pro Tip:** If Colab crashes with "Out of Memory" *during* the GGUF export step:
    1.  Don't panic. Your adapters are likely already saved/pushed during training.
    2.  Restart the Runtime (`Runtime > Restart Session`).
    3.  **Do not** run the training cell again.
    4.  Run the setup cells (install unsloth), load the model + adapters, and *then* run the GGUF cell alone.

### 4. 🎉 The Celebration
Once that GGUF file hits your Hugging Face repo:
1.  **Tweet/Post it:** "Just built Sheikh-Max: A thinking coding agent on a Colab T4!"
2.  **Tag me:** If you post on GitHub/socials, feel free to tag or share the link.

**Go ahead. Hit `Runtime > Run All`. Bismillah! 🚀**